In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import matplotlib.pyplot as plt
import torchvision
import numpy as np

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

In [ ]:
train_dataset = datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 176kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.25MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 29.3MB/s]


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 10)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 79.3MB/s]


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"✅ Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")


✅ Epoch [1/20], Loss: 0.1821


✅ Epoch [2/20], Loss: 0.1371


✅ Epoch [3/20], Loss: 0.1057


✅ Epoch [4/20], Loss: 0.0836


✅ Epoch [5/20], Loss: 0.0659


...

✅ Epoch [16/20], Loss: 0.0152


✅ Epoch [17/20], Loss: 0.0171


✅ Epoch [18/20], Loss: 0.0146


✅ Epoch [19/20], Loss: 0.0151


✅ Epoch [20/20], Loss: 0.0132


In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')
precision_cls, recall_cls, f1_cls, _ = precision_recall_fscore_support(all_labels, all_preds, average=None)

print("\n=== Метрики (в среднем по всем классам) ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\n=== Метрики по каждому классу ===")
for i, (p, r, f) in enumerate(zip(precision_cls, recall_cls, f1_cls)):
    print(f"Класс {i}:  Precision={p:.4f}, Recall={r:.4f}, F1={f:.4f}")


=== Метрики (в среднем по всем классам) ===
Accuracy:  0.9469
Precision: 0.9471
Recall:    0.9469
F1-score:  0.9466

=== Метрики по каждому классу ===
Класс 0:  Precision=0.8696, Recall=0.9140, F1=0.8913
Класс 1:  Precision=0.9980, Recall=0.9900, F1=0.9940
Класс 2:  Precision=0.9325, Recall=0.9260, F1=0.9293
Класс 3:  Precision=0.9589, Recall=0.9340, F1=0.9463
Класс 4:  Precision=0.9045, Recall=0.9750, F1=0.9384
Класс 5:  Precision=0.9960, Recall=0.9850, F1=0.9904
Класс 6:  Precision=0.8749, Recall=0.7970, F1=0.8341
Класс 7:  Precision=0.9768, Recall=0.9700, F1=0.9734
Класс 8:  Precision=0.9940, Recall=0.9950, F1=0.9945
Класс 9:  Precision=0.9656, Recall=0.9830, F1=0.9742


**Неплохое качество выдала самая простая модель В0**